# Preliminary Results

**Project title:** *No Breathing Room: Spatial Modeling of Air Pollution Exposure Along Jeepney Routes in Quezon City*

**Course and section: CS 138 WFY-2**

**Consultation date:**

## 1. Member Contributions and Signatures

| Member | Contributions | Signature |
| --- | --- | --- |
|ALBANO, Elliot Xander | | |
|CELESTINO, Leo | | |
|PACTOLIN, Renz Sergio | Added setup for getting weather data | |
|PADILLA, Klex Felouise|Proposal Summary| |
|SALTING, Ambernie Green | | |

## 2. Proposal Summary

### 2.1 Problem and Significance

This project investigates how commuting conditions relate to estimated exposure to fine particulate matter (PM2.5). Exposure during a trip depends on the pollutant concentrations encountered and the time spent along different road segments. Thus, a shorter or faster trip may not necessarily have the lowest estimated exposure. Research on commuting exposure provides the motivation for examining these differences (Panchal et al., 2022).

The study focuses on selected jeepney routes in Quezon City. It proposes combining available air-quality observations, environmental information, and route characteristics to estimate pollution along these corridors and simulate how concentrations change under simplified wind and emission conditions. The intended contribution is a computational proof of concept for comparing route exposure and examining the assumptions behind those estimates. Its outputs will be interpreted as model-based estimates, rather than direct measurements of passengers' inhaled dose or a validated air-quality forecasting service.

### 2.2 Research Questions and Hypotheses

The study retains the two research questions stated in the revised proposal:

1. How do route characteristics and environmental conditions influence estimated commuter exposure to air pollution, and which factors are most strongly associated with the estimated exposure?
2. How do pollutant concentrations evolve spatially and temporally under different wind and pollutant-source scenarios, and how can the resulting concentration estimates be used to compare commuting routes?

The following working hypotheses make the expected relationships explicit:

- **H1:** For otherwise comparable trips, routes that involve more time in areas with higher PM2.5 concentrations will have greater estimated cumulative exposure. Differences in concentration along a route may affect comparisons based on travel time or distance alone.
- **H2:** Changing wind conditions or pollutant-source intensity will change the simulated spatial and temporal distribution of PM2.5, potentially changing the relative exposure estimates for selected routes or route segments.

### 2.3 Scope, Study Area, and Jeepney Routes

The pollutant of interest is PM2.5, and the proposed study area covers selected commuting corridors in Quezon City. The revised proposal identifies three jeepney routes as areas of interest using Sakay.ph (By Implication Inc., n.d.):

- Katipunan to the UP Diliman campus, and vice versa;
- SM Fairview to Philcoa, and vice versa;
- SM North to the UP Diliman campus, and vice versa.

These are candidate study corridors. The final geographic extent, observation period, time resolution, and routes retained for analysis will depend on usable data coverage. Because the three routes serve different endpoints, comparisons across them will describe exposure under specified trip conditions.

The proposal identifies Quezon City air-quality resources and OpenAQ as possible sources of numerical PM2.5 observations, ERA5 as a possible source of weather information, and MMDA traffic-volume records as inputs to the pollutant-source model.

If monitoring coverage is insufficient, the study will narrow its domain or use explicitly labelled simplified simulations. If traffic data or a justified conversion from traffic volume to emissions is unavailable, pollutant sources will represent hypothetical scenarios. Interpretation will remain limited by monitoring coverage, route and travel-time information, and the assumptions of the two-dimensional model.

## 3. Stuff We Tried and Want to Try

### 3.1 Data Acquisition and Preparation

In [ ]:
%pip install requests pandas

In [ ]:
from __future__ import annotations
from pathlib import Path
import pandas as pd
from requests import get
from pandas import DataFrame
from dataclasses import dataclass


layout:
- email correspondence with CCESD
- FOI requested

#### OpenAQ: June 2026 daily PM2.5

Load the saved CSV in `OpenAQ-06-2026`; no API key or OpenAQ request is needed when running this notebook. The file already contains daily averages from location **4526988**, Near Katipunan Ave., Quezon City. Do not average these rows again. To refresh the download, run `retrieve_june.py` from inside that directory, then rerun the loading cell below.

Use the **start** of each reporting period in Asia/Manila as its calendar date: the interval June 30 to July 1 belongs to June 30. Keep `hours_observed` and `coverage_percent` alongside each mean so incomplete days remain identifiable.

In [ ]:
# Support kernels started in the repository, its parent, or the data directory.
project_candidates = [
    Path.cwd(),
    Path.cwd() / 'cs138-capstone-wfy2',
    Path.cwd().parent,
]
csv_relative = Path('OpenAQ-06-2026/openaq_4526988_june_2026_daily_pm25.csv')
project_dir = next(
    (root for root in project_candidates if (root / csv_relative).is_file()),
    None,
)
if project_dir is None:
    raise FileNotFoundError(f'Cannot find {csv_relative} from {Path.cwd()}')

pm25_raw = pd.read_csv(project_dir / csv_relative)
pm25_daily = pm25_raw.rename(columns={'pm25': 'pm25_mean'}).copy()
for column in ['datetime_from', 'datetime_to']:
    pm25_daily[column] = (
        pd.to_datetime(pm25_daily[column], utc=True).dt.tz_convert('Asia/Manila')
    )
pm25_daily['date'] = pm25_daily['datetime_from'].dt.strftime('%Y-%m-%d')
pm25_daily['location_id'] = 4526988
pm25_daily = pm25_daily.loc[
    pm25_daily['date'].between('2026-06-01', '2026-06-30')
].sort_values(['date', 'sensor_id']).reset_index(drop=True)
if pm25_daily.duplicated(['sensor_id', 'date']).any():
    raise ValueError('Expected one daily average per sensor and date.')

print(f'Loaded {len(pm25_daily)} daily PM2.5 observations.')
print(pm25_daily[[
    'date', 'sensor_id', 'pm25_mean', 'units',
    'hours_observed', 'coverage_percent',
]].head().to_string(index=False))

### 3.2 Spatiotemporal IDW and k-d Tree Search

### 3.3 Numerical Linear Algebra Formulation and Methods

### 3.4 Advection-Diffusion PDE and Finite Difference Scheme

### 3.5 Route Exposure Estimation and Comparison

### 3.6 Unsuccessful Attempts, Obstacles, and Planned Experiments

## 4. Results with Data Commentary

### 4.1 Computational Setup and Reproducibility

### 4.2 Data Availability, Quality, and Preprocessing

In [ ]:
# Summarize coverage per sensor without filling missing observations with zero.
pm25_coverage = pm25_daily.groupby('sensor_id').agg(
    days_present=('date', 'nunique'),
    days_with_pm25=('pm25_mean', 'count'),
    min_hours_observed=('hours_observed', 'min'),
    max_hours_observed=('hours_observed', 'max'),
    min_coverage_percent=('coverage_percent', 'min'),
    max_coverage_percent=('coverage_percent', 'max'),
)
pm25_coverage['missing_days'] = 30 - pm25_coverage['days_present']
print(pm25_coverage.to_string())

The saved June dataset contains 30 daily observations from sensor **13153272**, with 4 to 23 observed hours per day and reported coverage of 17% to 96%. All dates are present, but no daily mean has a complete 24 hours of observations. The loading step preserves all daily means and their coverage; no missing hours are replaced with zero and no coverage cutoff is imposed.

These are PM2.5 observations close to Katipunan Avenue. Spatial interpolation across the study corridors still requires additional monitoring locations and their coordinates (hopefully to be provided by CCESD).

### 4.3 IDW Pollution Maps and Interpolation Evaluation

### 4.4 Numerical Linear Algebra Results and Diagnostics

### 4.5 PDE Simulation Setup: Parameters, Initial Conditions, and Boundary Conditions

### 4.6 PDE Results: Wind and Source Scenarios

In [ ]:

@dataclass(frozen=True)
class WindData:
    wind_speed: float
    wind_direction: float

url = "https://archive-api.open-meteo.com/v1/archive"

# QC Circle
params = {
    "latitude": 14.6507,
    "longitude": 121.0494,
    "start_date": "2026-01-01",
    "end_date": "2026-06-30",
    "daily": [
        "wind_speed_10m_mean",
        "wind_direction_10m_dominant"
    ],
    "hourly": [
        "wind_speed_10m",
        "wind_direction_10m"
    ],
    "wind_speed_unit": "ms",
    "timezone": "Asia/Manila"
}

response = get(url, params=params)
response.raise_for_status()
data = response.json()

##
## DAILY DATA (to csv)
df_daily = DataFrame(data["daily"])
df_daily = df_daily.rename(columns={
    "time": "date",
    "wind_speed_10m_mean": "wind_speed",
    "wind_direction_10m_dominant": "wind_direction"
})
df_daily.to_csv("qc_memorial_circle_daily_wind.csv", index=False)

##
## HOURLY DATA (to csv)
df_hourly = DataFrame(data["hourly"])
df_hourly = df_hourly.rename(columns={
    "time": "timestamp",
    "wind_speed_10m": "wind_speed",
    "wind_direction_10m": "wind_direction"
})
df_hourly.to_csv("qc_memorial_circle_hourly_wind.csv", index=False)

##
##  DICTS
daily_weather: dict[str, WindData] = {
    row.date: WindData(row.wind_speed, row.wind_direction)
    for row in df_daily.itertuples(index=False)
}

hourly_weather: dict[tuple[str, str], WindData] = {
    tuple(row.timestamp.split("T")): WindData(
        row.wind_speed,
        row.wind_direction
    )
    for row in df_hourly.itertuples(index=False)
}


## DEBUG: Example Accesses
print(daily_weather["2026-01-01"])
print(hourly_weather[("2026-01-01", "00:00")])


#### Align daily PM2.5 with daily wind

Run the weather cell above after the OpenAQ loading cell in Section 3.1. Both datasets use Manila calendar dates. The left join below keeps every PM2.5 observation and selects its corresponding daily wind record from the longer weather period. This aligns dates between the Katipunan sensor and QC Memorial Circle weather series; the observations refer to different locations. The resulting `pm25_weather_daily` table is available for subsequent analysis.

In [ ]:
weather_daily = df_daily[['date', 'wind_speed', 'wind_direction']].copy()
weather_daily['date'] = pd.to_datetime(weather_daily['date']).dt.strftime('%Y-%m-%d')
pm25_weather_daily = pm25_daily.merge(
    weather_daily, on='date', how='left', validate='many_to_one',
)
missing_wind = pm25_weather_daily[['wind_speed', 'wind_direction']].isna().any(axis=1)
print(f'PM2.5 rows without complete daily wind data: {missing_wind.sum()}')
print(pm25_weather_daily[[
    'date', 'pm25_mean', 'units', 'coverage_percent',
    'wind_speed', 'wind_direction',
]].head().to_string(index=False))

### 4.7 Numerical Stability, Accuracy, and Convergence

### 4.8 Jeepney Route Exposure Comparisons

### 4.9 Overall Data Commentary: Successful and Unsuccessful Results, Assumptions, and Limitations

## 5. Questions for the Instructor

### Question 1

### Question 2

## References

The following references are carried over from the revised proposal for Section 2.

- Alvarez-Vazquez, L., Garcia-Chan, N., Martinez, A., & Vazquez-Mendez, M. (2017). Numerical simulation of air pollution due to traffic flow in urban networks. *Journal of Computational and Applied Mathematics, 326*, 44-61. https://doi.org/10.1016/j.cam.2017.05.017
- By Implication Inc. (n.d.). *Jeepney routes*. Sakay.ph. Retrieved October 29, 2026, from https://explore.sakay.ph/jeeps
- Li, L., Losser, T., Yorke, C., & Piltner, R. (2014). Fast inverse distance weighting-based spatiotemporal interpolation: A web-based application of interpolating daily fine particulate matter PM2.5 in the contiguous U.S. using parallel programming and k-d tree. *International Journal of Environmental Research and Public Health, 11*(9), 9101-9141. https://doi.org/10.3390/ijerph110909101
- Mahadi, N. N. I., & Puzi, S. M. (2022). Numerical solutions of advection-diffusion equation using explicit finite difference method. *Proceedings of Science and Mathematics, 11*, 136-143. https://science.utm.my/procscimath/wp-content/uploads/sites/605/2022/10/136-143-Nurul-Nur-Izzaty_Shazirawati-Mohd-Puzi.pdf
- Panchal, R., Panagi, M., May, H. R., Obszynska, J. A., Evans, M. S., Hansell, A. L., Gulliver, J., & Hey, J. D. V. (2022). Personal air pollution exposure during morning commute car and active transport journeys. *Journal of Transport & Health, 26*, 101365. https://doi.org/10.1016/j.jth.2022.101365

## LLM Usage Declaration

Padilla, Klex Felouise:

1. Python Notebook Initial Template
- LLM: Codex
- Purpose: Generating initial Python Notebook template
- Prompt: https://chatgpt.com/s/cx_6af14145dc588191a8a5e2dbafd3df32
- Output: `Preliminary_Results_Template.ipynb`. It has since been edited for brevity in this Python notebook (`cs138-wfy2-preliminaryresults.ipynb`).

2. OpenAQ Data Retrieval Assistance
- LLM: Codex
- Purpose: Retrieving PM2.5 data for June 2026, aggregated into daily averages, and integrating the saved CSV with coverage summaries and daily weather data in this notebook
- Prompt: https://chatgpt.com/s/cx_6afee7b4842481919c5e9a63edee9ec2
- Output: `OpenAQ-06-2026/retrieve_june.py` and OpenAQ loading, coverage, and weather-alignment cells in `cs138-wfy2-preliminaryresults.ipynb`